# RL-RAG on Kaggle T4 — end-to-end runner

Walks a Kaggle T4 (16 GB, Linux, CUDA) from a cold start to a trained R0+GRPO checkpoint with a `metrics.json`.

## Before you run — Kaggle settings (right-hand sidebar)

1. **Accelerator → GPU T4 x2** (we only use one; P100 is sm_60 which modern vLLM doesn't support — pick T4).
2. **Internet → ON** (required for `pip install`, `git clone`, HuggingFace download).
3. **Persistence → Files only** is fine. Nothing in `/kaggle/working/` survives session end unless you **Save Version** (menu top-right).
4. If the GitHub repo is **private**: Add-ons → Secrets → Add secret, label `GITHUB_TOKEN`, paste a Personal Access Token with `repo` scope. Toggle it ON for this notebook.

## Kaggle-specific differences from the Colab version
- No Google Drive. Everything lives under `/kaggle/working/` (20 GB writable, saved on Save Version) and `/kaggle/temp/` (ephemeral but fast).
- No idle disconnect during active cell execution, but notebook stops when you close the tab. For a 4–8 h training run, **Save Version** with "Run all" from the menu — Kaggle runs it to completion in the background.
- **30 h GPU/week** quota. One full training run costs ~1/5 of your weekly budget. Be deliberate.
- HF cache does not persist across sessions unless you snapshot it into a Kaggle Dataset. For a single-run workflow this doesn't matter; for repeated runs it's worth doing later.

## What this notebook does
1. Pull GITHUB_TOKEN from Kaggle Secrets if present.
2. Clone the repo into `/kaggle/working/rl-rag`.
3. Install torch cu121 + HF stack + vLLM + verl at the GRACE-pinned commit.
4. Verify CUDA + show GPU.
5. Download HotpotQA (tiny + full).
6. Build BM25 retrieval + prompts.
7. Baseline-eval the un-trained Qwen3-0.6B.
8. Smoke-test the verl stack with `configs/skeleton_t4.yaml`.
9. Train R0+GRPO with `configs/r0_grpo_t4.yaml` (200 steps, checkpoints every 25).
10. Evaluate the final checkpoint on the val split.
11. Compare baseline vs trained (Week-2 Gate PASS/FAIL).

## 1. Pull GitHub token (private repo only; skipped silently if no secret)

In [ ]:
import os
GITHUB_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    GITHUB_TOKEN = UserSecretsClient().get_secret('GITHUB_TOKEN')
    print('GITHUB_TOKEN loaded from Kaggle Secrets')
except Exception as e:
    print('No GITHUB_TOKEN secret found; assuming public repo. Reason:', type(e).__name__)

## 2. Clone (or pull) the repo into `/kaggle/working/rl-rag`

Kaggle's working dir is 20 GB and persists only when you hit **Save Version**. The repo itself is ~10 MB; the HF cache + training checkpoints are what fill it.

In [ ]:
import os, subprocess, pathlib

REPO_OWNER = 'elderflamevandle'
REPO_NAME  = 'RL-MSML635-Group-Project'
BRANCH     = 'feat/eval-failure-analysis-lj-gate'
REPO_LOCAL = '/kaggle/working/rl-rag'

if GITHUB_TOKEN:
    REPO_URL = f'https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git'
else:
    REPO_URL = f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git'

p = pathlib.Path(REPO_LOCAL)
if (p / '.git').exists():
    print(f'repo exists at {p}; pulling latest')
    subprocess.run(['git', '-C', str(p), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(p), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(p), 'pull', '--ff-only'], check=True)
else:
    print(f'cloning into {p}')
    p.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(p)], check=True)

%cd {REPO_LOCAL}
!git log -1 --oneline

## 3. Env vars (HF cache under `/kaggle/working`)

Keeps Qwen3-0.6B + any downloaded models inside the notebook snapshot so a Save Version preserves them.

In [ ]:
import os
os.environ['HF_HOME']             = '/kaggle/working/.hfcache'
os.environ['HF_HUB_CACHE']        = '/kaggle/working/.hfcache/hub'
os.environ['TRANSFORMERS_CACHE']  = '/kaggle/working/.hfcache/hub'
os.environ['PYTHONPATH']          = '/kaggle/working/rl-rag'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.makedirs(os.environ['HF_HUB_CACHE'], exist_ok=True)
for k in ['HF_HOME','HF_HUB_CACHE','TRANSFORMERS_CACHE','PYTHONPATH','TOKENIZERS_PARALLELISM']:
    print(f'{k}={os.environ[k]}')

## 4. Install deps (~5–8 min)

The `scripts/setup_colab.sh` script is platform-agnostic — it installs torch cu121, HF stack, vLLM 0.6.6, and verl. Works identically on Kaggle.

In [ ]:
!bash scripts/setup_colab.sh

## 5. CUDA sanity

Must print `Tesla T4 sm_75`. If it says `P100 sm_60` go back to the sidebar and switch accelerator to **GPU T4 x2** — vLLM 0.6.6 won't work on P100.

In [ ]:
!nvidia-smi
!python -c "import torch; p=torch.cuda.get_device_properties(0); print(p.name, 'sm_'+str(p.major)+str(p.minor), f'{p.total_memory/1024**3:.1f}GB')"

## 6. HotpotQA data

In [ ]:
!python scripts/download_hotpot_tiny.py
!python scripts/download_hotpot_full.py
!ls -lh data/raw/

## 7. Build retrieval cache (BM25 only — CPU, ~1 min)

In [ ]:
!python scripts/build_retrieval.py --no-dense
!ls -lh data/cache/
!echo '--- first row of bm25_train.jsonl ---'
!python -c "import json; row=json.loads(open('data/cache/bm25_train.jsonl').readline()); print('keys:', list(row)); assert 'prompt' in row, 'prompt field missing — verl will fail'; print('prompt starts:', row['prompt'][:200])"

## 8. Baseline eval (pre-RL metrics)

In [ ]:
!python -m src.eval.evaluate \n    --checkpoint Qwen/Qwen3-0.6B \n    --split data/cache/bm25_val.jsonl \n    --out results/baseline_t4/metrics.json \n    --backend hf --device cuda --max-new-tokens 256 \n    --max-samples 50
!echo '--- baseline metrics ---'
!cat results/baseline_t4/metrics.json

## 9a. Smoke test — 1-step verl run with the T4 skeleton config

First-run failure is expected: verl will likely complain about a YAML field name. Read the error, edit `configs/skeleton_t4.yaml`, re-run this cell. **Do not proceed to step 9b until this passes.**

In [ ]:
!bash scripts/launch_run.sh configs/skeleton_t4.yaml

## 9b. Full R0+GRPO training (200 steps, ~4–8 h on T4)

Checkpoints every 25 steps. If the Kaggle session ends, re-run this cell — `resume_mode: auto` in the config picks up where it left off.

**For a long training run:** use the menu **Save Version → Run all → Save**. Kaggle runs the whole notebook to completion in the background, even if you close the tab. This is the Kaggle-equivalent of "let Colab run overnight".

In [ ]:
!bash scripts/launch_run.sh configs/r0_grpo_t4.yaml

## 10. Evaluate the trained checkpoint → `metrics.json`

In [ ]:
import glob, os, pathlib, subprocess

run_dirs = sorted(glob.glob('results/r0_grpo_qwen3-0.6b_s0_*'))
assert run_dirs, 'No r0_grpo_* run dir found under results/ — did step 9b complete?'
run_dir = run_dirs[-1]
print('run_dir:', run_dir)

ckpts = sorted(glob.glob(f'{run_dir}/checkpoints/global_step_*'))
assert ckpts, f'No checkpoints under {run_dir}/checkpoints/ — training did not save'
ckpt = ckpts[-1]
print('checkpoint:', ckpt)

out = f'{run_dir}/metrics.json'
cmd = [
    'python', '-m', 'src.eval.evaluate',
    '--checkpoint', ckpt,
    '--split', 'data/cache/bm25_val.jsonl',
    '--out', out,
    '--backend', 'vllm',
    '--device', 'cuda',
    '--max-new-tokens', '384',
    '--bertscore',
]
print('running:', ' '.join(cmd))
subprocess.run(cmd, check=True)
print('---')
print(open(out).read())

## 11. Compare baseline vs trained (Week-2 Gate)

In [ ]:
import json, glob

baseline = json.load(open('results/baseline_t4/metrics.json'))
run_dir  = sorted(glob.glob('results/r0_grpo_qwen3-0.6b_s0_*'))[-1]
trained  = json.load(open(f'{run_dir}/metrics.json'))

keys = ['balanced_accuracy', 'em_answerable', 'abstention_unanswerable']
print(f'{"metric":<28} {"baseline":>10} {"trained":>10} {"delta":>10}')
for k in keys:
    b = baseline.get(k, float('nan'))
    t = trained.get(k, float('nan'))
    print(f'{k:<28} {b:>10.3f} {t:>10.3f} {t-b:>+10.3f}')

print()
print('Week-2 Gate: trained balanced_accuracy must be in [0.71, 0.91].')
ba = trained['balanced_accuracy']
status = 'PASS' if 0.71 <= ba <= 0.91 else 'FAIL'
print(f'  balanced_accuracy = {ba:.3f}  →  {status}')

## 12. Download the results folder

`/kaggle/working/` is zipped and attached to the notebook version when you Save Version. To get just the metrics file to your laptop quickly, use the **Output** panel on the right after a Save Version, or run the cell below to list what's there.

In [ ]:
!find results -maxdepth 3 -type f \( -name 'metrics.json' -o -name 'train.log' -o -name 'predictions.jsonl' \) | xargs -I {} ls -lh {}